<a href="https://colab.research.google.com/github/LsYoli/fisica_computacional/blob/main/Taller_6/taller6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Comprueba gfortran (y lo instala en Colab si falta) y crea la carpeta del Taller
!which gfortran > /dev/null || (apt-get -qq update && apt-get -qq install -y gfortran > /dev/null)
!gfortran --version | head -1
!mkdir -p taller_nucleo_atomico

GNU Fortran (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## 9. Taller 6: Fórmula semiempírica de masa atómica

### A. Enunciado y requisitos

**Contexto físico.** La masa de un núcleo con $Z$ protones y $A$ nucleones es menor que la suma de las masas de sus componentes: la diferencia es la energía de enlace. En el modelo de la gota líquida (Bethe y Weizsäcker), la energía de enlace **por nucleón** es la suma de cuatro términos:

$$E_B(Z,A) = \underbrace{-\,a_V}_{\text{volumen}} \; + \; \underbrace{a_S\,A^{-1/3}}_{\text{superficie}} \; + \; \underbrace{a_C\,Z^2A^{-4/3}}_{\text{Coulomb}} \; + \; \underbrace{a_A\,\frac{(Z - A/2)^2}{A^2}}_{\text{asimetría}}$$

donde $Z$ es el número de protones, $A$ el número de nucleones (protones más neutrones) y $N = A - Z$ el número de neutrones. Cada término tiene un origen físico:

| Término | Expresión | Origen físico | Efecto |
|---|---|---|---|
| Volumen | $-a_V$ | La fuerza nuclear fuerte es de corto alcance: cada nucleón solo interactúa con sus vecinos inmediatos, así que el enlace es proporcional al número de nucleones y **por nucleón es constante**. | Liga ($<0$) |
| Superficie | $a_S\,A^{-1/3}$ | Los nucleones de la superficie tienen menos vecinos. El núcleo es una esfera de radio $R \propto A^{1/3}$; su superficie es $\propto A^{2/3}$ y, dividida entre $A$, da $A^{-1/3}$. | Resta enlace; pesa más en núcleos livianos |
| Coulomb | $a_C\,Z^2A^{-4/3}$ | Repulsión electrostática entre los $Z$ protones, repartidos en la esfera de radio $R \propto A^{1/3}$: la energía total es $\propto Z^2/A^{1/3}$ y, dividida entre $A$, da $Z^2A^{-4/3}$. | Resta enlace; pesa más en núcleos pesados |
| Asimetría | $a_A\,(Z - A/2)^2/A^2$ | Por el principio de exclusión de Pauli, protones y neutrones ocupan niveles de energía separados: este término por sí solo favorece $Z = N = A/2$, y cualquier exceso de uno de los dos cuesta energía. El mínimo del modelo completo depende también de los otros términos. | Resta enlace si $Z \neq A/2$ |

y la masa atómica aproximada se calcula mediante

$$M(Z,A) = Z\,M_H + (A - Z)\,M_n + A\,E_B(Z,A),$$

donde $Z\,M_H$ es la masa de $Z$ átomos de hidrógeno (cada uno, un protón y un electrón), $(A - Z)\,M_n$ la masa de los $N$ neutrones libres, y $A\,E_B$ la energía de enlace **total**, que es negativa y reduce la masa respecto de los componentes separados. Como se utiliza $M_H$, el resultado incluye la masa de los $Z$ electrones: corresponde aproximadamente a una **masa atómica**, despreciando sus energías de enlace electrónicas.

En las ecuaciones se adopta $c=1$: los símbolos de masa representan su energía equivalente $Mc^2$, expresada en **MeV**. Para expresar la masa como tal, la unidad correspondiente es MeV/$c^2$.

Con esta convención, $E_B < 0$ para un núcleo ligado: cuanto más negativo, más ligado. Las tablas experimentales suelen dar la energía de enlace por nucleón con signo positivo, $B/A = -E_B$.

**Constantes** (O. Zubairi y F. Weber, *Introduction to Computational Physics for Undergraduates*, 2.ª ed., IOP, 2024, problema 7.2):

| $a_V$ | $a_S$ | $a_C$ | $a_A$ | $M_n$ | $M_H$ |
|---|---|---|---|---|---|
| 15.8 MeV | 18.3 MeV | 0.71 MeV | 92.7 MeV | 939.57 MeV | 938.78 MeV |
| coeficiente de volumen | coeficiente de superficie | coeficiente de Coulomb | coeficiente de asimetría | masa del neutrón | masa del átomo de hidrógeno |

Los cuatro coeficientes no se deducen de una teoría: se ajustan a las masas medidas de muchos núcleos. Por eso el modelo se llama *semiempírico*.

**Preguntas que debe responder el programa.**

1. ¿Cuánto vale cada término, la energía de enlace por nucleón y la masa atómica aproximada de los núclidos ${}^{244}$Pu, ${}^{70}$Ge, ${}^{179}$Hf, ${}^{38}$Ar, ${}^{56}$Fe, ${}^{40}$Ca, ${}^{235}$U y ${}^{238}$U?
2. Para cada número de nucleones $A$, ¿qué número de protones $Z$ da el núcleo más ligado? El conjunto de estos puntos define la **curva de máximo enlace del modelo**, relacionada con el valle de estabilidad.
3. ¿Cuál es el núcleo con mayor enlace **por nucleón** en toda esa curva, para el intervalo de $A$ estudiado?

Para un $A$ fijo, se busca el núcleo de menor $E_B(Z,A)$ entre $Z = 1, 2, \dots, A$. En todo el taller, «más ligado» se refiere a mayor enlace **por nucleón**, es decir, a un $E_B$ más negativo.

**Alcance físico de la búsqueda.** Minimizar $E_B$ a $A$ fijo no equivale exactamente a minimizar $M$, porque $M_H$ y $M_n$ son diferentes. La estabilidad frente a transformaciones beta requiere comparar las masas de los isóbaros y considerar los canales de decaimiento. En este taller se conserva únicamente la búsqueda del máximo enlace: **no debe cambiarse el criterio de búsqueda por una minimización de la masa**. Para mantener los nombres de archivo y la arquitectura común, se conservan `valle.dat` y el procedimiento `Z_estable`, entendido aquí como el $Z$ de máximo enlace según el modelo.

**Diseño previo (obligatorio)**

En la sección B.1 se entrega un **pseudocódigo marco**: muestra qué procedimientos tiene el programa, cómo se conectan y en qué orden se ejecuta. Antes de escribir código, complete sus tres espacios marcados, como en el Taller 1:

- **[COMPLETE 1]:** cómo se encuentra, para un $A$ dado, el $Z$ de menor $E_B$.
- **[JUSTIFIQUE]:** por qué corresponde usar una subrutina para entregar los dos resultados separados, de acuerdo con la regla de diseño de la sección 3.3.
- **[COMPLETE 2]:** cómo se identifica, durante el ciclo sobre $A$, el núcleo más ligado de toda la curva.

El pseudocódigo completado es un entregable. Escríbalo en español estructurado, no en Fortran. Si su programa final se aparta de él, explique la diferencia debajo del pseudocódigo.

**Requisitos del programa de Fortran**

| | Requisito |
|---|---|
| R1 | Todos los archivos del Taller (código y resultados) van en la carpeta `taller_nucleo_atomico/`. |
| R2 | Las constantes y los procedimientos del modelo están en un **módulo**, en el archivo `modulo_nucleo.f90`. El programa principal está en otro archivo, `programa_nucleo.f90`. |
| R3 | Las constantes son `parameter`. No hay variables globales que no sean constantes. |
| R4 | Cada uno de los cuatro términos es una función. La energía de enlace por nucleón es otra función, que usa las cuatro anteriores; la masa, otra más. |
| R5 | Un procedimiento recibe $A$ y entrega dos resultados: el $Z$ de máximo enlace y su $E_B$. Justifique el uso de una subrutina según la regla de la sección 3.3. |
| R6 | Todos los argumentos de todos los procedimientos tienen `intent`. |
| R7 | Todos los reales son `real(dp)`, con `dp = kind(1.0d0)`. |

**Archivos que debe escribir el programa de Fortran** (columnas separadas por espacios, energías en MeV con al menos 8 decimales, masa con al menos 4):

| Archivo | Filas | Columnas |
|---|---|---|
| `taller_nucleo_atomico/nucleos.dat` | 8, en el orden de la pregunta 1 | $Z$, $A$, $E_{\rm vol}$, $E_{\rm sup}$, $E_{\rm Coul}$, $E_{\rm asim}$, $E_B$, $M$ |
| `taller_nucleo_atomico/valle.dat` | una por cada $A = 4, 5, \dots, 260$ | $A$, $Z$ de máximo enlace, $E_B$ de ese núcleo |
| `taller_nucleo_atomico/maximo.dat` | 1 | $A$, $Z$ y $E_B$ del núcleo más ligado de `valle.dat` |

**Requisitos de la parte de Python**

| | Requisito |
|---|---|
| P1 | Las funciones de lectura y análisis están en el módulo `taller_nucleo_atomico/analisis_nucleo.py`, que se importa desde el notebook. |
| P2 | Una gráfica de $B/A = -E_B$ contra $A$ a lo largo de la curva de máximo enlace, con título, ejes con unidades, leyenda y los valores experimentales de la sección C, guardada en `taller_nucleo_atomico/curva_enlace.png`. |
| P3 | El núcleo más ligado, encontrado en Python a partir de `valle.dat`, escrito en `taller_nucleo_atomico/maximo_python.dat` con el mismo formato de `maximo.dat`. |

**Actividades**

1. Complete el pseudocódigo marco (sección B.1).
2. Escriba, compile y ejecute el programa de Fortran (secciones B.2 y B.3: dos archivos).
3. Escriba el módulo de Python y úselo para la gráfica y el máximo (secciones C.1 a C.4: un módulo y una celda de uso).
4. Ejecute la celda de verificación (sección D). Esta celda no debe modificarse. La entrega debe dar **4/4 PASS**. Esto es necesario, pero no suficiente: también se revisarán el diseño, el código, la gráfica, la interpretación y la sustentación.
5. Responda las preguntas de interpretación (sección E).
6. Elabore el reporte de una página (sección F).

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

No se entrega un esqueleto del programa: la organización en procedimientos es parte de lo que se evalúa. Las secciones 2 a 6 contienen todas las construcciones necesarias. En la sustentación, cada estudiante explicará su propio código.

</div>

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Construya el programa por partes. Escriba primero el módulo con las funciones de los cuatro términos y pruébelas con el ${}^{244}$Pu, cuyo resultado está en la tabla de la sección D. Solo cuando ese caso coincida, escriba el procedimiento del $Z$ de máximo enlace y el ciclo sobre $A$.

</div>

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite la división entera.** En Fortran, `-1/3` es una división entre enteros y vale `0`; entonces `a**(-1/3)` vale `1` para cualquier `a`, sin ningún aviso. Escriba los exponentes como reales (`-1.0_dp/3.0_dp`) y convierta los enteros con `real(a, dp)` antes de elevarlos. El mismo cuidado aplica a `A/2` en el término de asimetría cuando $A$ es impar.

</div>

### B. Programa de Fortran

#### B.1 Diseño: pseudocódigo marco

La celda siguiente contiene el marco del programa. Edítela (doble clic) y reemplace cada espacio entre corchetes por su pseudocódigo. No cambie el resto del marco.

```
MÓDULO del modelo
    constantes: a_V, a_S, a_C, a_A, M_n, M_H
    funciones de los cuatro términos               (entradas: Z y/o A; salida: el término [MeV])
    función E_B(Z, A)                              (usa las cuatro funciones anteriores)
    función M(Z, A)                                (usa E_B)
    procedimiento Z_estable                        (entrada: A; resultados: el Z de máximo enlace y su E_B)
       Inicializar Z_min = 1 y E_min = E_B(1, A)
        Para Z = 2, 3, ..., A:
            calcular E = E_B(Z, A)
            si E < E_min: guardar Z_min = Z y E_min = E
        Entregar Z_min y E_min
        Se usa una subrutina porque el procedimiento entrega dos resultados (Z y E_B).
        Una función solo devuelve un valor por su nombre; según la regla de la sección 3.3,
        varios resultados se entregan con argumentos intent(out) en una subrutina.

PROGRAMA PRINCIPAL (usa el módulo)
    para cada uno de los 8 núcleos, en el orden del enunciado:
        escribir una fila de nucleos.dat: Z, A, los cuatro términos, E_B y M
    para A = 4, 5, ..., 260:
        obtener el Z de máximo enlace y su E_B con Z_estable
        escribir una fila de valle.dat: A, Z, E_B
        si E_B < E_mejor:
            guardar A_mejor = A, Z_mejor = Z y E_mejor = E_B
```

#### B.2 Qué debe preparar

La parte B consiste en **dos archivos de Fortran**, que se escriben en las dos celdas de la sección B.3, que ya empiezan con la línea `%%writefile` y el nombre del archivo. La tercera celda, ya escrita, los compila y ejecuta; no la modifique.

| Archivo | Qué contiene | Requisitos |
|---|---|---|
| `taller_nucleo_atomico/modulo_nucleo.f90` | El **módulo** del modelo: <br>• las seis constantes de la tabla del enunciado; <br>• una función para cada uno de los cuatro términos; <br>• una función para $E_B(Z,A)$, que use las cuatro anteriores; <br>• una función para $M(Z,A)$; <br>• el procedimiento que, dado $A$, entrega el $Z$ de máximo enlace y su $E_B$. | R2 a R7 |
| `taller_nucleo_atomico/programa_nucleo.f90` | El **programa principal**: usa el módulo y escribe los tres archivos de resultados. Si necesita procedimientos propios (por ejemplo, para escribir una fila), van después de su `contains`. | R2, R6, R7 |

**Resultados de la parte B**, que escribe el programa al ejecutarse, en la misma carpeta y con el formato de la tabla de la sección A: `nucleos.dat`, `valle.dat` y `maximo.dat`.

**La parte B está terminada cuando** la celda de compilación se ejecuta sin errores, los tres archivos existen y la primera fila de `nucleos.dat` (el ${}^{244}$Pu) tiene $E_B \approx -7.5361$ MeV. Esto es un control inicial; la validación completa se realiza en D.

#### B.3 Código de Fortran

In [2]:
%%writefile taller_nucleo_atomico/modulo_nucleo.f90
! Taller 6 -- Módulo de la fórmula semiempírica de masa
module modulo_nucleo
  implicit none

  ! --- Constantes globales (solo constantes, R3) ---
  integer, parameter  :: dp  = kind(1.0d0)
  real(dp), parameter :: a_V = 15.8_dp      ! coeficiente de volumen [MeV]
  real(dp), parameter :: a_S = 18.3_dp      ! coeficiente de superficie [MeV]
  real(dp), parameter :: a_C = 0.71_dp      ! coeficiente de Coulomb [MeV]
  real(dp), parameter :: a_A = 92.7_dp      ! coeficiente de asimetría [MeV]
  real(dp), parameter :: M_n = 939.57_dp    ! masa del neutrón [MeV]
  real(dp), parameter :: M_H = 938.78_dp    ! masa del átomo de hidrógeno [MeV]

contains

  ! ------------------------------------------------------------------
  ! termino_volumen: término de volumen, -a_V
  !   Entradas: ninguna;  Salida: término [MeV]
  ! ------------------------------------------------------------------
  real(dp) function termino_volumen()
    termino_volumen = -a_V
  end function termino_volumen

  ! ------------------------------------------------------------------
  ! termino_superficie: a_S * A^(-1/3)
  !   Entrada: A (nucleones);  Salida: término [MeV]
  ! ------------------------------------------------------------------
  real(dp) function termino_superficie(A)
    integer, intent(in) :: A
    termino_superficie = a_S*real(A, dp)**(-1.0_dp/3.0_dp)
  end function termino_superficie

  ! ------------------------------------------------------------------
  ! termino_coulomb: a_C * Z^2 * A^(-4/3)
  !   Entradas: Z (protones), A (nucleones);  Salida: término [MeV]
  ! ------------------------------------------------------------------
  real(dp) function termino_coulomb(Z, A)
    integer, intent(in) :: Z, A
    termino_coulomb = a_C*real(Z, dp)**2*real(A, dp)**(-4.0_dp/3.0_dp)
  end function termino_coulomb

  ! ------------------------------------------------------------------
  ! termino_asimetria: a_A * (Z - A/2)^2 / A^2
  !   Entradas: Z (protones), A (nucleones);  Salida: término [MeV]
  ! ------------------------------------------------------------------
  real(dp) function termino_asimetria(Z, A)
    integer, intent(in) :: Z, A
    termino_asimetria = a_A*(real(Z, dp) - real(A, dp)/2.0_dp)**2/real(A, dp)**2
  end function termino_asimetria

  ! ------------------------------------------------------------------
  ! energia_enlace: energía de enlace por nucleón E_B(Z,A)
  !   Entradas: Z, A;  Salida: E_B [MeV] (negativa si el núcleo está ligado)
  ! ------------------------------------------------------------------
  real(dp) function energia_enlace(Z, A)
    integer, intent(in) :: Z, A
    energia_enlace = termino_volumen() + termino_superficie(A) &
                   + termino_coulomb(Z, A) + termino_asimetria(Z, A)
  end function energia_enlace

  ! ------------------------------------------------------------------
  ! masa_atomica: M(Z,A) = Z*M_H + (A-Z)*M_n + A*E_B
  !   Entradas: Z, A;  Salida: masa atómica aproximada [MeV]
  ! ------------------------------------------------------------------
  real(dp) function masa_atomica(Z, A)
    integer, intent(in) :: Z, A
    masa_atomica = real(Z, dp)*M_H + real(A - Z, dp)*M_n &
                 + real(A, dp)*energia_enlace(Z, A)
  end function masa_atomica

  ! ------------------------------------------------------------------
  ! Z_estable: Z de máximo enlace (menor E_B) para un A dado
  !   Entrada: A;  Salidas: Z_max, E_max [MeV]
  !   Es subrutina porque entrega dos resultados (sección 3.3).
  ! ------------------------------------------------------------------
  subroutine Z_estable(A, Z_max, E_max)
    integer, intent(in)   :: A
    integer, intent(out)  :: Z_max
    real(dp), intent(out) :: E_max
    integer  :: Z_prueba           ! contador local
    real(dp) :: E_prueba           ! energía local

    Z_max = 1
    E_max = energia_enlace(1, A)
    do Z_prueba = 2, A
       E_prueba = energia_enlace(Z_prueba, A)
       if (E_prueba < E_max) then
          Z_max = Z_prueba
          E_max = E_prueba
       end if
    end do
  end subroutine Z_estable

end module modulo_nucleo

Writing taller_nucleo_atomico/modulo_nucleo.f90


In [5]:
%%writefile taller_nucleo_atomico/programa_nucleo.f90
! Taller 6 -- Programa principal: escribe nucleos.dat, valle.dat y maximo.dat
program programa_nucleo
  use modulo_nucleo
  implicit none

  integer  :: A, Z, u_nuc, u_valle, u_max
  integer  :: A_mejor, Z_mejor
  real(dp) :: E_B, E_mejor

  ! Los 8 núclidos, en el orden del enunciado: Pu-244, Ge-70, Hf-179, Ar-38,
  ! Fe-56, Ca-40, U-235, U-238 (se guardan como constantes, sin arreglos)
  ! --- nucleos.dat ---
  open(newunit=u_nuc, file='nucleos.dat', status='replace')
  call escribe_fila(u_nuc, 94, 244)
  call escribe_fila(u_nuc, 32,  70)
  call escribe_fila(u_nuc, 72, 179)
  call escribe_fila(u_nuc, 18,  38)
  call escribe_fila(u_nuc, 26,  56)
  call escribe_fila(u_nuc, 20,  40)
  call escribe_fila(u_nuc, 92, 235)
  call escribe_fila(u_nuc, 92, 238)
  close(u_nuc)

  ! --- valle.dat y búsqueda del núcleo más ligado ---
  open(newunit=u_valle, file='valle.dat', status='replace')
  E_mejor = 0.0_dp                  ! todo E_B de la curva es negativo
  A_mejor = 0
  Z_mejor = 0
  do A = 4, 260
     call Z_estable(A, Z, E_B)
     write(u_valle, '(2i6, f18.10)') A, Z, E_B
     if (E_B < E_mejor) then
        A_mejor = A
        Z_mejor = Z
        E_mejor = E_B
     end if
  end do
  close(u_valle)

  ! --- maximo.dat ---
  open(newunit=u_max, file='maximo.dat', status='replace')
  write(u_max, '(2i6, f18.10)') A_mejor, Z_mejor, E_mejor
  close(u_max)

  print '(a, i4, a, i3, a, f12.8, a)', 'Núcleo más ligado: A =', A_mejor, &
        '  Z =', Z_mejor, '  E_B =', E_mejor, ' MeV'

contains

  ! ------------------------------------------------------------------
  ! escribe_fila: escribe una fila de nucleos.dat
  !   Entradas: unidad (unidad del archivo), Z_n, A_n
  !   Salida:   ninguna; escribe Z, A, los cuatro términos, E_B y M
  !   Es subrutina porque realiza una acción (escribir un archivo).
  ! ------------------------------------------------------------------
  subroutine escribe_fila(unidad, Z_n, A_n)
    integer, intent(in) :: unidad, Z_n, A_n
    write(unidad, '(2i6, 5f18.10, f18.6)') Z_n, A_n, &
          termino_volumen(), termino_superficie(A_n), &
          termino_coulomb(Z_n, A_n), termino_asimetria(Z_n, A_n), &
          energia_enlace(Z_n, A_n), masa_atomica(Z_n, A_n)
  end subroutine escribe_fila

end program programa_nucleo

Overwriting taller_nucleo_atomico/programa_nucleo.f90


In [6]:
# Compilación y ejecución (NO MODIFICAR): primero el módulo, después el programa.
# Se ejecuta dentro de taller_nucleo_atomico/ para que los .dat queden allí.
!cd taller_nucleo_atomico && gfortran -Wall -Wextra modulo_nucleo.f90 programa_nucleo.f90 -o nucleo && ./nucleo

Núcleo más ligado: A =  65  Z = 29  E_B = -8.69506906 MeV


### C. Análisis en Python

#### C.1 Qué debe preparar

La parte C consiste en **un módulo de Python** y **una celda que lo usa**. Trabaja solo con los archivos que escribió la parte B: no vuelve a calcular la fórmula.

| Dónde | Qué contiene | Requisitos |
|---|---|---|
| `taller_nucleo_atomico/analisis_nucleo.py` (celda C.3) | Cuatro funciones, cada una con su docstring de entradas y salidas: <br>• una que lea `valle.dat` y devuelva sus tres columnas; <br>• una que encuentre el núcleo más ligado por nucleón recorriendo los datos con un ciclo y un condicional, como en Fortran (sin `min`, `np.min` ni `np.argmin`); <br>• una que escriba ese núcleo en `maximo_python.dat`, con el formato de `maximo.dat`; <br>• una que haga la gráfica de $B/A = -E_B$ contra $A$ con los puntos experimentales de la celda C.2 y la guarde como imagen. | P1 a P3 |
| Celda de uso (C.4) | Importa el módulo (las primeras líneas ya están escritas), llama a las cuatro funciones e imprime el núcleo más ligado. | — |

**Resultados de la parte C**: `taller_nucleo_atomico/maximo_python.dat` y `taller_nucleo_atomico/curva_enlace.png`.

**La parte C está terminada cuando** Python imprime el mismo $A$ y $Z$ que `maximo.dat` y la gráfica muestra la curva del modelo con los diez puntos experimentales.

#### C.2 Datos experimentales

Valores de la energía de enlace por nucleón, $B/A = -E_B$, en MeV (evaluación de masas atómicas AME2020). Use esta celda tal como está. Los puntos representan los núclidos indicados; la curva representa el máximo enlace para cada $A$ según el modelo. No son necesariamente el mismo $Z$, por lo que la comparación gráfica es orientativa. Para una comparación numérica de un núclido concreto, use su mismo $Z$ y $A$.

In [7]:
# Datos experimentales: (símbolo, Z, A, B/A [MeV]) -- AME2020
EXPERIMENTALES = [
    ("He-4",    2,   4, 7.0739),
    ("C-12",    6,  12, 7.6801),
    ("O-16",    8,  16, 7.9762),
    ("Ca-40",  20,  40, 8.5513),
    ("Fe-56",  26,  56, 8.7903),
    ("Ni-62",  28,  62, 8.7945),
    ("Sn-120", 50, 120, 8.5045),
    ("Pb-208", 82, 208, 7.8675),
    ("U-235",  92, 235, 7.5909),
    ("U-238",  92, 238, 7.5701),
]

#### C.3 Módulo de Python

In [8]:
%%writefile taller_nucleo_atomico/analisis_nucleo.py
"""Taller 6 -- Módulo de análisis de los resultados de Fortran."""
import numpy as np
import matplotlib
matplotlib.use("Agg")                      # guarda la imagen sin abrir ventana
import matplotlib.pyplot as plt

# Datos experimentales: (símbolo, Z, A, B/A [MeV]) -- AME2020
EXPERIMENTALES = [
    ("He-4",    2,   4, 7.0739),
    ("C-12",    6,  12, 7.6801),
    ("O-16",    8,  16, 7.9762),
    ("Ca-40",  20,  40, 8.5513),
    ("Fe-56",  26,  56, 8.7903),
    ("Ni-62",  28,  62, 8.7945),
    ("Sn-120", 50, 120, 8.5045),
    ("Pb-208", 82, 208, 7.8675),
    ("U-235",  92, 235, 7.5909),
    ("U-238",  92, 238, 7.5701),
]

def leer_valle(archivo):
    """Lee valle.dat.

    Entrada: nombre del archivo
    Salida:  tupla (A, Z, E_B), cada una una columna (arreglo de NumPy)
    """
    datos = np.loadtxt(archivo)
    return datos[:, 0], datos[:, 1], datos[:, 2]

def nucleo_mas_ligado(A, Z, E_B):
    """Núcleo con menor E_B (mayor enlace por nucleón), con ciclo y condicional.

    Entradas: columnas A, Z, E_B
    Salida:   tupla (A_mejor, Z_mejor, E_mejor)
    """
    A_mejor, Z_mejor, E_mejor = A[0], Z[0], E_B[0]
    for a, z, e in zip(A, Z, E_B):
        if e < E_mejor:
            A_mejor, Z_mejor, E_mejor = a, z, e
    return A_mejor, Z_mejor, E_mejor

def escribir_maximo(archivo, A_mejor, Z_mejor, E_mejor):
    """Escribe el núcleo más ligado con el mismo formato de maximo.dat.

    Entradas: nombre del archivo y los tres valores
    Salida:   ninguna (escribe el archivo)
    """
    with open(archivo, "w") as f:
        f.write(f"{int(A_mejor):6d}{int(Z_mejor):6d}{E_mejor:18.10f}\n")

def graficar_curva(A, E_B, experimentales, archivo):
    """Grafica B/A = -E_B contra A, con los puntos experimentales.

    Entradas: columnas A y E_B, lista de experimentales, nombre de la imagen
    Salida:   ninguna (guarda el PNG)
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(A, -E_B, "-", color="#162A54", label="Modelo (máximo enlace)")
    A_exp = [fila[2] for fila in experimentales]
    BA_exp = [fila[3] for fila in experimentales]
    ax.plot(A_exp, BA_exp, "o", color="#C9972C", label="Experimental (AME2020)")
    ax.set_xlabel("Número de nucleones $A$")
    ax.set_ylabel("Energía de enlace por nucleón $B/A$ [MeV]")
    ax.set_title("Curva de máximo enlace: modelo de la gota líquida")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(archivo, dpi=150)
    plt.close(fig)

Writing taller_nucleo_atomico/analisis_nucleo.py


#### C.4 Uso del módulo

In [9]:
# C.4 -- Uso del módulo de análisis
import sys, importlib
sys.path.insert(0, "taller_nucleo_atomico")
import analisis_nucleo
_ = importlib.reload(analisis_nucleo)

carpeta = "taller_nucleo_atomico/"
A, Z, E_B = analisis_nucleo.leer_valle(carpeta + "valle.dat")
A_m, Z_m, E_m = analisis_nucleo.nucleo_mas_ligado(A, Z, E_B)
analisis_nucleo.escribir_maximo(carpeta + "maximo_python.dat", A_m, Z_m, E_m)
analisis_nucleo.graficar_curva(A, E_B, analisis_nucleo.EXPERIMENTALES,
                               carpeta + "curva_enlace.png")
print(f"Núcleo más ligado (Python): A = {A_m:.0f}, Z = {Z_m:.0f}, E_B = {E_m:.8f} MeV")

Núcleo más ligado (Python): A = 65, Z = 29, E_B = -8.69506906 MeV


### D. Verificación

La celda siguiente revisa los archivos, sus dimensiones, valores numéricos y coherencia. Los resultados de Fortran deben ser posteriores a sus dos archivos fuente. `maximo_python.dat` debe ser posterior al módulo de análisis y a `valle.dat`. Los cuatro controles informan PASS o FAIL sin detener el notebook. **Esta celda no debe modificarse.**

| Caso | Qué se comprueba |
|---|---|
| 1 | Los ocho núclidos de `nucleos.dat`: orden, cuatro términos, $E_B$ y masa atómica aproximada. |
| 2 | `valle.dat` completo: $A=4,\dots,260$, valores enteros válidos y máximo enlace para cada $A$; incluye la referencia $A=208$, $Z=82$. |
| 3 | Referencia $A=238$, $Z=92$, y coherencia entre el $E_B$ guardado y el modelo en todas las filas de `valle.dat`. |
| 4 | Los máximos de Fortran y Python coinciden con el mínimo global de `valle.dat`; existe una imagen PNG no vacía. |

Las referencias se evalúan en doble precisión con las constantes del enunciado. Por ello pueden diferir en los últimos decimales de una salida publicada con otra precisión. Para ${}^{244}$Pu: $E_B \approx -7.53611738$ MeV y $M \approx 227\,342.0074$ MeV.

**4/4 PASS es necesario, pero no suficiente.** La celda no evalúa la arquitectura del código, el uso de `intent`, la independencia de variables locales, la calidad de la gráfica ni las explicaciones. No consulte su implementación como guía para resolver el algoritmo: utilice los mensajes para localizar qué resultado debe revisar.


In [10]:
# ===================== VERIFICACIÓN — NO MODIFICAR =====================
from pathlib import Path
import numpy as np

CARPETA = Path("taller_nucleo_atomico")
veredictos = []

def informar(nombre, condicion, detalle=""):
    veredictos.append(bool(condicion))
    print(f"{'PASS' if condicion else 'FAIL'} | {nombre}" + (f" — {detalle}" if detalle else ""))

def leer_salida(nombre, filas, columnas, origen="fortran"):
    ruta = CARPETA / nombre
    if not ruta.is_file():
        raise ValueError(f"falta {ruta}")
    if origen == "fortran":
        fuentes = [CARPETA / "modulo_nucleo.f90", CARPETA / "programa_nucleo.f90"]
    else:
        fuentes = [CARPETA / "analisis_nucleo.py", CARPETA / "valle.dat"]
    for fuente in fuentes:
        if not fuente.is_file():
            raise ValueError(f"falta {fuente}")
        if ruta.stat().st_mtime < fuente.stat().st_mtime:
            raise ValueError(f"{nombre} es anterior a {fuente.name}: vuelva a ejecutar el cálculo correspondiente")
    datos = np.loadtxt(ruta, ndmin=2)
    if datos.shape != (filas, columnas) or not np.isfinite(datos).all():
        raise ValueError(f"formato incorrecto en {nombre}: se esperaban {filas} filas y {columnas} columnas finitas")
    return datos

# Implementación de referencia exclusiva de la verificación.
def referencia_terminos(z, a):
    z, a = np.asarray(z, dtype=float), np.asarray(a, dtype=float)
    return np.broadcast_arrays(-15.8 + np.zeros_like(a), 18.3*a**(-1.0/3.0),
                               0.71*z**2*a**(-4.0/3.0), 92.7*(z-a/2.0)**2/a**2)

def referencia_enlace(z, a):
    return sum(referencia_terminos(z, a))

def leer_valle():
    datos = leer_salida("valle.dat", 257, 3)
    if not np.array_equal(datos[:, 0], np.arange(4, 261)):
        raise ValueError("valle.dat debe tener A = 4, 5, ..., 260, en ese orden")
    if not np.array_equal(datos[:, 1], np.rint(datos[:, 1])):
        raise ValueError("los valores de Z deben ser enteros")
    if np.any((datos[:, 1] < 1) | (datos[:, 1] > datos[:, 0])):
        raise ValueError("cada fila debe cumplir 1 <= Z <= A")
    return datos

# 1) Ocho núclidos: todos los términos, el enlace y la masa.
try:
    nuc = leer_salida("nucleos.dat", 8, 8)
    orden = np.array([(94,244), (32,70), (72,179), (18,38), (26,56), (20,40), (92,235), (92,238)])
    if not np.array_equal(nuc[:, :2], orden):
        raise ValueError("Z y A deben ser enteros y seguir el orden del enunciado")
    z, a = orden[:, 0], orden[:, 1]
    terminos = np.column_stack(referencia_terminos(z, a))
    eb = terminos.sum(axis=1)
    masa = z*938.78 + (a-z)*939.57 + a*eb
    ok = (np.allclose(nuc[:, 2:6], terminos, rtol=0, atol=1e-7)
          and np.allclose(nuc[:, 6], eb, rtol=0, atol=1e-7)
          and np.allclose(nuc[:, 7], masa, rtol=0, atol=1e-3)
          and np.allclose(nuc[:, 2:6].sum(axis=1), nuc[:, 6], rtol=0, atol=1e-7))
    informar("Caso 1: ocho núclidos", ok, f"Pu-244: E_B = {nuc[0,6]:.8f} MeV; M = {nuc[0,7]:.4f} MeV")
except Exception as error:
    informar("Caso 1: ocho núclidos", False, str(error))

# 2) La búsqueda obtiene el máximo enlace para cada A del intervalo.
try:
    valle = leer_valle()
    correctos = []
    for a, z, eb in valle:
        candidatos = np.arange(1, int(a)+1)
        energias = referencia_enlace(candidatos, a)
        correctos.append(abs(float(referencia_enlace(z, a)) - energias.min()) <= 1e-7)
    fila = valle[valle[:, 0] == 208][0]
    ok = all(correctos) and fila[1] == 82 and abs(fila[2] - (-7.800561449221129)) <= 1e-7
    informar("Caso 2: máximo enlace para cada A", ok, f"{sum(correctos)}/257 búsquedas correctas; A=208: Z={fila[1]:.0f}")
except Exception as error:
    informar("Caso 2: máximo enlace para cada A", False, str(error))

# 3) Energías de todas las filas y referencia de U-238.
try:
    valle = leer_valle()
    esperado = referencia_enlace(valle[:, 1], valle[:, 0])
    fila = valle[valle[:, 0] == 238][0]
    ok = (np.allclose(valle[:, 2], esperado, rtol=0, atol=1e-7)
          and fila[1] == 92 and abs(fila[2] - (-7.579586363043683)) <= 1e-7)
    informar("Caso 3: energías de la curva y U-238", ok, f"A=238: Z={fila[1]:.0f}; E_B={fila[2]:.8f} MeV")
except Exception as error:
    informar("Caso 3: energías de la curva y U-238", False, str(error))

# 4) Mínimo global: dos lenguajes y archivo de imagen.
try:
    valle = leer_valle()
    mf = leer_salida("maximo.dat", 1, 3)[0]
    mp = leer_salida("maximo_python.dat", 1, 3, origen="python")[0]
    referencia = valle[np.argmin(valle[:, 2])]
    def coincide(fila):
        return (np.array_equal(fila[:2], referencia[:2])
                and abs(fila[2]-referencia[2]) <= 1e-7)
    imagen = CARPETA / "curva_enlace.png"
    png_ok = (imagen.is_file() and imagen.stat().st_size > 8
              and imagen.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n")
    ok = (coincide(mf) and coincide(mp)
          and abs(referencia[2] - (-8.69506905766211)) <= 1e-7 and png_ok)
    informar("Caso 4: mínimo global e imagen", ok,
             f"Fortran: A={mf[0]:.0f}, Z={mf[1]:.0f}; Python: A={mp[0]:.0f}, Z={mp[1]:.0f}; PNG: {'presente' if png_ok else 'falta o formato incorrecto'}")
except Exception as error:
    informar("Caso 4: mínimo global e imagen", False, str(error))

print(f"\nResumen: {sum(veredictos)}/4 PASS")
print("La arquitectura, los requisitos y la interpretación se revisan por separado.")


PASS | Caso 1: ocho núclidos — Pu-244: E_B = -7.53611738 MeV; M = 227342.0074 MeV
PASS | Caso 2: máximo enlace para cada A — 257/257 búsquedas correctas; A=208: Z=82
PASS | Caso 3: energías de la curva y U-238 — A=238: Z=92; E_B=-7.57958636 MeV
PASS | Caso 4: mínimo global e imagen — Fortran: A=65, Z=29; Python: A=65, Z=29; PNG: presente

Resumen: 4/4 PASS
La arquitectura, los requisitos y la interpretación se revisan por separado.


### E. Preguntas de interpretación

Responda en celdas de texto, debajo de cada pregunta. Use los resultados de su propio programa. Los cálculos adicionales de esta sección pueden realizarse llamando a los procedimientos de sus módulos; **no cambie el modelo de cuatro términos ni los archivos que utiliza la verificación de D**.

1. Usando `nucleos.dat`, elija un núcleo relativamente liviano y otro pesado e identifíquelos. Compare los términos de superficie y Coulomb en ambos casos. ¿Cuál de esos dos términos penaliza más el enlace en cada uno? Explique cómo su competencia contribuye a que la curva $B/A$ presente un máximo.

Se eligieron el ${}^{40}$Ca (liviano) y el ${}^{244}$Pu (pesado), con los valores de `nucleos.dat`:

| Núcleo | Superficie [MeV] | Coulomb [MeV] | Más penalizante |
|---|---|---|---|
| ${}^{40}$Ca ($Z=20$) | 5.3510 | 2.0761 | Superficie |
| ${}^{244}$Pu ($Z=94$) | 2.9286 | 4.1146 | Coulomb |

En el calcio, que es liviano, una fracción grande de los nucleones está en la superficie y tiene menos vecinos, así que ese término pesa más. En el plutonio, la repulsión entre 94 protones crece como $Z^2$ y el término de Coulomb pasa a ser el que más penaliza.

La superficie baja con $A$ (va como $A^{-1/3}$) y Coulomb sube (casi como $A^{2/3}$). Al principio, al aumentar $A$, mejora el enlace porque la superficie pierde peso. Después Coulomb crece más rápido y el enlace empeora. De esa competencia sale el máximo de la curva $B/A$.

2. El modelo predice su mayor enlace por nucleón en un $A$ distinto del ${}^{62}$Ni experimental. Calcule la diferencia de $E_B$ entre el máximo del modelo y el punto de su curva con $A=62$. Exprese también la diferencia como porcentaje de la magnitud de $E_B$ en ese punto. Discuta el tamaño de la diferencia y las limitaciones del modelo que podrían contribuir a ella. Distinga esta comparación entre puntos de la curva de una comparación directa con el núclido experimental: para esta última debe usar $Z=28$, $A=62$.

El máximo del modelo está en $A=65$, $Z=29$, con $E_B=-8.695069$ MeV. En $A=62$ el $Z$ de máximo enlace es 28, con $E_B=-8.690891$ MeV.

- Diferencia: $-8.695069-(-8.690891)=-0.004178$ MeV
- En porcentaje: $0.004178/8.690891\approx0.048\ \%$

La diferencia es muy pequeña: la curva es casi plana cerca del máximo, así que $A=62$ queda prácticamente empatado con $A=65$.

Comparación directa con el núclido experimental (${}^{62}$Ni, $Z=28$, $A=62$): el modelo da $B/A=-E_B=8.6909$ MeV y el valor experimental es $8.7945$ MeV. La diferencia es de $0.1036$ MeV, cerca del $1.2\ \%$ del valor experimental. Esta diferencia es más grande que la anterior y mide qué tanto falla el modelo, no la posición del máximo.

Entre las limitaciones del modelo están que no incluye el apareamiento ni los efectos de capas (números mágicos), que sus cuatro coeficientes son un ajuste promedio y que trata al núcleo como una gota uniforme.*Escriba aquí su respuesta y los resultados que la sustentan.*

3. Para $A=56$, ¿qué $Z$ tiene el mayor enlace por nucleón en el modelo de cuatro términos? Compárelo con el hierro ($Z=26$): calcule la diferencia de energía **total**, indicando el orden de la resta, y diga cuál resulta más ligado.

   La fórmula completa incluye un quinto término, el de **apareamiento**: protones y neutrones tienden a formar pares. En la energía total, la corrección vale $-a_P A^{-1/2}$ si $Z$ y $N$ son ambos pares, $+a_P A^{-1/2}$ si ambos son impares, y cero si $A$ es impar, con $a_P\approx12$ MeV. Calcule la corrección para los dos núcleos comparados y determine si invierte su orden de enlace. Recuerde que esta corrección es de energía **total**: para sumarla a $E_B$ debe dividirla entre $A$.

   Este es un **cálculo adicional de comparación**: no incorpore el apareamiento a las funciones del modelo base ni regenere con él los archivos de D. Basta comparar los dos candidatos; no se pide repetir la búsqueda global con el modelo ampliado.

En el modelo de cuatro términos, para $A=56$ el $Z$ de mayor enlace por nucleón es $Z=25$ (con $N=31$), no el del hierro.

| | $E_B$ [MeV] | Energía total $A\,E_B$ [MeV] |
|---|---|---|
| $Z=25$ | $-8.679530$ | $-486.0537$ |
| $Z=26$ (Fe) | $-8.658320$ | $-484.8659$ |

Diferencia de energía total (Z = 25 menos Z = 26): $-486.0537-(-484.8659)=-1.1877$ MeV. El $Z=25$ resulta más ligado.

Con apareamiento, $a_P A^{-1/2}=12/\sqrt{56}=1.6036$ MeV:

- $Z=25$, $N=31$ (ambos impares): corrección $+1.6036$ MeV, energía total $-484.4501$ MeV.
- $Z=26$, $N=30$ (ambos pares): corrección $-1.6036$ MeV, energía total $-486.4695$ MeV.

Nueva diferencia (Z = 25 menos Z = 26): $-484.4501-(-486.4695)=+2.0194$ MeV. El orden se invierte: ahora el hierro ($Z=26$) queda más ligado, que es lo que se observa en la naturaleza.

4. Reutilizando las funciones de su módulo, calcule la energía liberada en la fisión simétrica idealizada ${}^{236}\mathrm{U}\to2\,{}^{118}\mathrm{Pd}$ ($Z=46$): $Q=M(92,236)-2M(46,118)$. Las masas se utilizan aquí como energías equivalentes en MeV. Como ambos lados tienen el mismo número total de protones, las masas electrónicas incluidas se cancelan en esta diferencia, dentro de la aproximación adoptada. Si $Q>0$, el canal es energéticamente favorable. Compárelo con la escala típica de unos 200 MeV por fisión del uranio y explique por qué este canal idealizado no reproduce todos los productos ni el reparto de energía de una fisión real.

Con las funciones del módulo:

- $M(92,236)=219\,873.8753$ MeV
- $M(46,118)=109\,847.8415$ MeV

$$Q=M(92,236)-2\,M(46,118)=178.19\ \text{MeV}$$

Como $Q>0$, el canal es energéticamente favorable. El valor queda por debajo de los cerca de 200 MeV típicos de una fisión del uranio, pero está en la misma escala.

Este canal idealizado no reproduce una fisión real por varias razones:

- En la realidad el reparto casi nunca es simétrico: salen fragmentos de masas distintas, con una distribución de productos.
- Se liberan neutrones, que aquí no aparecen.
- Los fragmentos quedan muy ricos en neutrones y siguen decayendo después, liberando más energía.
- La energía no queda toda en la diferencia de masas: se reparte entre energía cinética de los fragmentos, neutrones y radiación.

El modelo de la gota líquida solo da la masa de dos núcleos concretos y no describe nada de esto.

### F. Entregables y reporte

**En el repositorio GitHub**, en la carpeta `Taller_6/`: este notebook ejecutado de principio a fin, con el pseudocódigo marco completado en la sección B.1, y la carpeta `taller_nucleo_atomico/` con `modulo_nucleo.f90`, `programa_nucleo.f90`, `analisis_nucleo.py`, los cuatro archivos `.dat` y `curva_enlace.png`.

**En el Campus Virtual**: el reporte de una página, con la plantilla LaTeX del curso, y el enlace a la carpeta del repositorio. El reporte debe incluir:

- **Contexto físico:** qué describe la fórmula semiempírica y qué representa la curva de máximo enlace calculada, distinguiéndola de la estabilidad frente al decaimiento.
- **Método:** cómo se organizó el programa (módulo, funciones, subrutina), apoyado en su pseudocódigo, y cómo se buscó el $Z$ de máximo enlace.
- **Resultado:** la gráfica `curva_enlace.png` y la comparación del modelo con dos valores experimentales, usando para cada comparación el mismo $Z$ y $A$. Indique las diferencias numéricas y la convención de signos.
- **Obstáculo o decisión de implementación:** por ejemplo, el orden de compilación, el uso de `intent` o el paso de enteros a reales en las potencias $A^{-1/3}$.
- **Conclusión:** incluya la respuesta a la pregunta E3.
- **Uso de herramientas de IA:** cuáles se usaron y para qué, si se usó alguna.

**Criterios de revisión**

Se valorarán la organización en procedimientos, el uso de argumentos e `intent`, las variables locales, la reutilización de funciones y la separación entre cálculo, salida y análisis. También se revisarán los resultados numéricos, la claridad de la gráfica, la interpretación física y la capacidad de explicar y modificar el propio código en la sustentación. Los PASS no reemplazan estos criterios.

**Antes de entregar**

- [ ] Los tres espacios del pseudocódigo marco (sección B.1) están completos.
- [ ] Reiniciar y ejecutar todas las celdas completa el flujo; solo aparecen los errores de compilación anunciados en los ejemplos didácticos. El código del Taller 6 compila y se ejecuta sin errores.
- [ ] La verificación de la sección D da 4/4 PASS.
- [ ] El código de Fortran cumple los requisitos R1 a R7, y el de Python, P1 a P3.
- [ ] Cada procedimiento tiene un comentario que dice qué calcula, con qué entradas y qué salidas.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**Evite** entregar archivos `.dat` de una ejecución anterior: la verificación compara su fecha con la del código y marca FAIL si el código cambió después. La comprobación también revisa que `maximo_python.dat` no sea anterior al módulo de Python ni a `valle.dat`. Antes de entregar, reinicie el entorno y ejecute todas las celdas (en Colab: reiniciar sesión y ejecutar todo; en Jupyter: `Restart & Run All`). Los ejemplos de errores anunciados son deliberados.

</div>